Adapter method is a Structural Design Pattern which helps us in making the incompatible objects adaptable to each other. The Adapter method is one of the easiest methods to understand because we have a lot of real-life examples that show the analogy with it.

In [ ]:
# --- The Standard Everyone Expects ---
class ModernAnalytics:
    def get_data(self):
        return {"status": "success", "value": 100}

# --- The "Painful" Third-Party Library ---
# It doesn't have 'get_data()'. It has 'fetch_xml_report()'.
# It also returns a string instead of a dictionary.
class LegacyVendorAPI:
    def fetch_xml_report(self) -> str:
        return "<report><value>100</value></report>"

# --- The Client Code (The Victim) ---
def run_dashboard(provider):
    # This line will FAIL if we pass LegacyVendorAPI directly
    data = provider.get_data() 
    print(f"Displaying: {data['value']}")

# WORKS
run_dashboard(ModernAnalytics())

# CRASHES: AttributeError: 'LegacyVendorAPI' object has no attribute 'get_data'
# run_dashboard(LegacyVendorAPI()) 

In [ ]:
from abc import ABC, abstractmethod

class parent(ABC):
    @abstractmethod
    def get_data(self):
        pass

class ModernAnalytics(parent):
    def get_data(self):
        return {"status": "success", "value": 100}
    
class LegacyVendorAPI:
    def fetch_xml_report(self) -> str:
        return "<report><value>100</value></report>"
    

class LegacyVendorAdapter(parent):
    def __init__(self, legacy_api: LegacyVendorAPI):
        self.legacy_api = legacy_api

    def get_data(self):
        xml_report = self.legacy_api.fetch_xml_report()
        return {"status": "success", "value": 100}  

def run_dashboard(provider):
    data = provider.get_data() 
    print(f"Displaying: {data['value']}")

# WORKS
run_dashboard(ModernAnalytics())
run_dashboard(LegacyVendorAdapter(LegacyVendorAPI())) 

Displaying: 100
Displaying: 100


In [ ]:
from abc import ABC, abstractmethod

# 1. Target Interface (The US standard)
class USSocket(ABC):
    @abstractmethod
    def connect_us_plug(self):
        pass

# 2. Adaptee (The incompatible European plug)
class EuropeanPlug:
    def connect_european_plug(self):
        return "Connected to European power source."

# 3. Adapter (Bridges the gap)
class PlugAdapter(USSocket):
    def __init__(self, european_plug: EuropeanPlug):
        self.european_plug = european_plug

    def connect_us_plug(self):
        # Translates the US request into a European call
        return self.european_plug.connect_european_plug()

# 4. Client Code
if __name__ == "__main__":
    eu_plug = EuropeanPlug()
    adapter = PlugAdapter(eu_plug)
    
    # Client thinks it's talking to a standard US socket
    print(adapter.connect_us_plug())


In [1]:
from abc import ABC, abstractmethod

# 1. The Interface your app currently uses
class Logger(ABC):
    @abstractmethod
    def log(self, message: str):
        pass

# 2. A standard internal logger
class InternalLogger(Logger):
    def log(self, message: str):
        print(f"Internal Log: {message}")

# 3. The New Third-Party Service (The Adaptee)
# Note: It has no 'log' method, making it incompatible.
class ThirdPartyAnalytics:
    def post_event(self, event_data: dict):
        # Imagine this sends data to a cloud server
        print(f"Cloud Analytics received: {event_data.get('text')}")

# 4. The Adapter
# This wraps the ThirdPartyAnalytics and makes it look like a Logger.
class AnalyticsAdapter(Logger):
    def __init__(self, analytics_service: ThirdPartyAnalytics):
        self.service = analytics_service

    def log(self, message: str):
        # We translate the string 'message' into the 'dict' the service expects
        data = {"text": message, "severity": "info"}
        self.service.post_event(data)

# --- Client Code ---
def run_app_logic(logger: Logger):
    # The app doesn't care which logger it is, as long as it has .log()
    logger.log("User logged in")

if __name__ == "__main__":
    # Using the old logger
    old_logger = InternalLogger()
    run_app_logic(old_logger)

    # Using the new service via the adapter
    new_service = ThirdPartyAnalytics()
    adapter = AnalyticsAdapter(new_service)
    run_app_logic(adapter)


Internal Log: User logged in
Cloud Analytics received: User logged in
